# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_04 — Hawkes Process Order Flow

### Research question

How can Hawkes processes model clustered order arrivals, buy/sell self-excitation, cross-excitation, order-flow imbalance, and short-horizon execution risk?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
06_04_hawkes_process_order_flow
```

The previous notebooks modelled execution schedules, dynamic execution policies, and square-root market impact. This notebook moves deeper into market microstructure:

> order flow is not independent through time.

Market orders, cancellations, quote updates, and aggressive buys/sells often cluster. A buy market order can increase the probability of more buy market orders shortly after. A sell burst can trigger more sells. Buy and sell flows can also excite each other through liquidity replenishment, adverse selection, or mean-reversion behaviour.

A Hawkes process captures this clustering.

It covers:

1. Poisson process baseline;
2. self-exciting point processes;
3. univariate Hawkes process;
4. multivariate buy/sell Hawkes process;
5. exponential kernels;
6. intensity dynamics;
7. branching ratio and stability;
8. Ogata thinning simulation;
9. synthetic order-flow generation;
10. buy/sell self- and cross-excitation;
11. order-flow imbalance;
12. price impact toy model;
13. likelihood for exponential Hawkes;
14. maximum-likelihood calibration;
15. branching matrix diagnostics;
16. residual time-change diagnostics;
17. intensity forecasting;
18. execution-risk use cases;
19. Hawkes-driven liquidity stress;
20. governance flags;
21. saved outputs and manifest.

The key idea:

> A Hawkes process turns order arrivals from independent random ticks into a feedback system where each event can temporarily increase the chance of future events.

## 1. Why Hawkes processes for order flow?

A simple Poisson process assumes arrivals are independent:

$$
N(t+\Delta t)-N(t)\sim Poisson(\lambda \Delta t)
$$

But order flow is clustered:

- trades arrive in bursts;
- aggressive buy orders can trigger more aggressive buys;
- sell pressure can persist;
- liquidity shocks can self-reinforce;
- quote activity can spike after trades.

A Hawkes process models this by allowing the intensity itself to jump after events:

$$
\begin{aligned}
\lambda(t) &= \mu \\
&\quad + \sum_{t_i<t} \alpha e^{-\beta(t-t_i)}
\end{aligned}
$$

where:

- $\mu$: baseline intensity;
- $\alpha$: excitation jump after an event;
- $\beta$: decay rate of excitation.

Each event temporarily raises the future event rate.

## 2. Multivariate Hawkes process

For buy and sell market orders, define two counting processes:

$$
N_B(t), \quad N_S(t)
$$

with intensities:

$$
\begin{aligned}
\lambda_B(t) &= \mu_B \\
&\quad + \sum_{t_k^B<t}\alpha_{BB}e^{-\beta_{BB}(t-t_k^B)} \\
&\quad + \sum_{t_k^S<t}\alpha_{BS}e^{-\beta_{BS}(t-t_k^S)}
\end{aligned}
$$

$$
\begin{aligned}
\lambda_S(t) &= \mu_S \\
&\quad + \sum_{t_k^B<t}\alpha_{SB}e^{-\beta_{SB}(t-t_k^B)} \\
&\quad + \sum_{t_k^S<t}\alpha_{SS}e^{-\beta_{SS}(t-t_k^S)}
\end{aligned}
$$

Interpretation:

- $\alpha_{BB}$: buys excite more buys;
- $\alpha_{SS}$: sells excite more sells;
- $\alpha_{BS}$: sells excite buys;
- $\alpha_{SB}$: buys excite sells.

The cross terms can represent mean-reversion, liquidity response, or feedback between sides.

## 3. Branching ratio and stability

For exponential kernels, the integrated excitation from process $j$ to process $i$ is:

$$
G_{ij} = \frac{\alpha_{ij}}{\beta_{ij}}
$$

The matrix $G$ is called the branching matrix.

The Hawkes process is stable if:

$$
\rho(G) < 1
$$

where $\rho(G)$ is the spectral radius.

Interpretation:

- $\rho(G)\approx 0$: close to Poisson;
- $\rho(G)\approx 0.5$: moderate clustering;
- $\rho(G)\to 1$: near-critical, bursty, unstable order flow.

For execution, near-critical order flow is dangerous because liquidity shocks can persist.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import minimize
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

SCIPY_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class HawkesOrderFlowConfig:
    horizon_seconds: float = 23_400.0
    seed: int = 42
    baseline_buy: float = 0.045
    baseline_sell: float = 0.043
    alpha_bb: float = 0.090
    alpha_bs: float = 0.040
    alpha_sb: float = 0.035
    alpha_ss: float = 0.085
    beta_bb: float = 1.40
    beta_bs: float = 1.20
    beta_sb: float = 1.25
    beta_ss: float = 1.35
    max_events: int = 60_000
    intensity_cap: float = 20.0
    price_tick: float = 0.01
    initial_mid: float = 100.0
    spread_ticks: int = 1
    permanent_impact_ticks: float = 0.015
    temporary_impact_ticks: float = 0.010
    imbalance_window_seconds: float = 60.0
    residual_bins: int = 40
    output_subdir: str = "hawkes_process_order_flow_v1"

config = HawkesOrderFlowConfig()
config

## 5. Parameter matrices and stability diagnostics

In [ ]:
def parameter_matrices(config):
    mu = np.array([config.baseline_buy, config.baseline_sell], dtype=float)

    alpha = np.array([
        [config.alpha_bb, config.alpha_bs],
        [config.alpha_sb, config.alpha_ss],
    ], dtype=float)

    beta = np.array([
        [config.beta_bb, config.beta_bs],
        [config.beta_sb, config.beta_ss],
    ], dtype=float)

    branching = alpha / beta
    eigvals = np.linalg.eigvals(branching)
    spectral_radius = float(np.max(np.abs(eigvals)))

    labels = ["BUY", "SELL"]

    return {
        "mu": mu,
        "alpha": alpha,
        "beta": beta,
        "branching": branching,
        "eigvals": eigvals,
        "spectral_radius": spectral_radius,
        "labels": labels,
    }

params = parameter_matrices(config)

branching_report = pd.DataFrame(
    params["branching"],
    index=params["labels"],
    columns=params["labels"],
)

stability_report = pd.DataFrame([{
    "spectral_radius": params["spectral_radius"],
    "stable": params["spectral_radius"] < 1.0,
    "eigenvalues": str(params["eigvals"]),
}])

branching_report, stability_report

## 6. Ogata thinning simulation for multivariate Hawkes

We simulate events with a thinning algorithm.

At each time:

1. compute total intensity $\Lambda(t)=\sum_i \lambda_i(t)$;
2. sample candidate waiting time from exponential distribution with rate upper bound;
3. accept the candidate with probability $\Lambda(t)/\bar{\Lambda}$;
4. choose event type according to relative intensities;
5. update intensity state.

For exponential kernels, the excitation state decays between events and jumps when an event occurs.

In [ ]:
def simulate_bivariate_hawkes(config, params):
    rng = np.random.default_rng(config.seed)

    mu = params["mu"]
    alpha = params["alpha"]
    beta = params["beta"]

    t = 0.0
    excitation = np.zeros(2, dtype=float)

    events = []
    intensity_rows = []

    last_t = 0.0

    for event_id in range(config.max_events):
        current_intensity = mu + excitation
        total_intensity = float(current_intensity.sum())

        if total_intensity <= 0:
            break

        lambda_bar = min(max(total_intensity * 1.25, total_intensity + 1e-8), config.intensity_cap)
        wait = rng.exponential(1.0 / lambda_bar)
        candidate_t = t + wait

        if candidate_t > config.horizon_seconds:
            break

        dt = candidate_t - t

        # Decay each excitation contribution approximately using row-specific average decay.
        # For exact multivariate exponential kernels we track each source-specific contribution below.
        t = candidate_t

        # Recompute excitation exactly from events within practical memory for clarity.
        excitation_exact = np.zeros(2)
        for past_t, past_type in events[-5000:]:
            lag = t - past_t
            if lag >= 0:
                excitation_exact += alpha[:, past_type] * np.exp(-beta[:, past_type] * lag)

        excitation = excitation_exact
        current_intensity = mu + excitation
        total_intensity = float(current_intensity.sum())

        if rng.uniform() <= total_intensity / lambda_bar:
            probs = current_intensity / total_intensity
            event_type = int(rng.choice([0, 1], p=probs))

            events.append((t, event_type))

            intensity_rows.append({
                "event_id": len(events) - 1,
                "time": t,
                "event_type": event_type,
                "event_name": "BUY" if event_type == 0 else "SELL",
                "lambda_buy_before": current_intensity[0],
                "lambda_sell_before": current_intensity[1],
                "total_intensity_before": total_intensity,
            })

        if len(events) >= config.max_events:
            break

    events_df = pd.DataFrame(intensity_rows)
    return events_df

events = simulate_bivariate_hawkes(config, params)

events.head(), events.shape

## 7. Event-flow diagnostics

We summarise:

- buy count;
- sell count;
- total count;
- buy ratio;
- events per second;
- interarrival times.

In [ ]:
def event_flow_diagnostics(events, config):
    counts = events["event_name"].value_counts()
    buy_count = int(counts.get("BUY", 0))
    sell_count = int(counts.get("SELL", 0))
    total_count = int(len(events))

    interarrival = events["time"].diff().dropna()

    summary = pd.DataFrame([{
        "buy_count": buy_count,
        "sell_count": sell_count,
        "total_count": total_count,
        "buy_ratio": buy_count / total_count if total_count else np.nan,
        "events_per_second": total_count / config.horizon_seconds,
        "mean_interarrival_seconds": interarrival.mean(),
        "median_interarrival_seconds": interarrival.median(),
        "p05_interarrival_seconds": interarrival.quantile(0.05),
        "p95_interarrival_seconds": interarrival.quantile(0.95),
    }])

    return summary, interarrival

flow_summary, interarrival = event_flow_diagnostics(events, config)

flow_summary.T

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(interarrival, bins=80)
plt.title("Interarrival Time Distribution")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(events["time"], np.arange(len(events)))
plt.title("Cumulative Order Arrivals")
plt.xlabel("Time, seconds")
plt.ylabel("Cumulative events")
plt.show()

## 8. Intensities through time

The Hawkes intensity should spike after bursts and decay between events.

In [ ]:
def compute_intensity_grid(events, params, horizon_seconds, grid_step=5.0):
    grid = np.arange(0, horizon_seconds + grid_step, grid_step)
    mu = params["mu"]
    alpha = params["alpha"]
    beta = params["beta"]

    event_times = events["time"].to_numpy()
    event_types = events["event_type"].to_numpy()

    rows = []
    for t in grid:
        lamb = mu.copy()
        mask = event_times < t
        for past_t, typ in zip(event_times[mask][-5000:], event_types[mask][-5000:]):
            lag = t - past_t
            lamb += alpha[:, typ] * np.exp(-beta[:, typ] * lag)

        rows.append({
            "time": t,
            "lambda_buy": lamb[0],
            "lambda_sell": lamb[1],
            "total_intensity": lamb.sum(),
            "intensity_imbalance": (lamb[0] - lamb[1]) / max(lamb.sum(), 1e-12),
        })

    return pd.DataFrame(rows)

intensity_grid = compute_intensity_grid(events, params, config.horizon_seconds, grid_step=10.0)

intensity_grid.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(intensity_grid["time"], intensity_grid["lambda_buy"], label="BUY intensity")
plt.plot(intensity_grid["time"], intensity_grid["lambda_sell"], label="SELL intensity")
plt.title("Hawkes Intensities Through Time")
plt.xlabel("Time, seconds")
plt.ylabel("Intensity")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(intensity_grid["time"], intensity_grid["intensity_imbalance"])
plt.axhline(0, linestyle="--")
plt.title("Intensity Imbalance")
plt.xlabel("Time, seconds")
plt.ylabel("(lambda_buy - lambda_sell) / total")
plt.show()

## 9. Poisson baseline comparison

A Poisson process with the same average rate has exponential interarrival times but no clustering.

We compare Hawkes against Poisson using:

- interarrival coefficient of variation;
- event count dispersion over windows;
- autocorrelation of event counts.

In [ ]:
def simulate_poisson_baseline(flow_summary, config):
    rng = np.random.default_rng(config.seed + 777)
    rate = flow_summary["events_per_second"].iloc[0]
    t = 0.0
    rows = []
    event_id = 0

    while t < config.horizon_seconds and event_id < config.max_events:
        t += rng.exponential(1.0 / rate)
        if t > config.horizon_seconds:
            break
        typ = int(rng.choice([0, 1]))
        rows.append({
            "event_id": event_id,
            "time": t,
            "event_type": typ,
            "event_name": "BUY" if typ == 0 else "SELL",
        })
        event_id += 1

    return pd.DataFrame(rows)

poisson_events = simulate_poisson_baseline(flow_summary, config)

def count_by_window(events, window_seconds):
    bins = np.arange(0, config.horizon_seconds + window_seconds, window_seconds)
    labels = bins[:-1]
    counts, _ = np.histogram(events["time"], bins=bins)
    return pd.Series(counts, index=labels)

hawkes_counts_60 = count_by_window(events, 60.0)
poisson_counts_60 = count_by_window(poisson_events, 60.0)

clustering_report = pd.DataFrame([
    {
        "process": "hawkes",
        "mean_count_60s": hawkes_counts_60.mean(),
        "var_count_60s": hawkes_counts_60.var(ddof=1),
        "dispersion_index": hawkes_counts_60.var(ddof=1) / hawkes_counts_60.mean(),
        "lag1_count_autocorr": hawkes_counts_60.autocorr(lag=1),
        "interarrival_cv": interarrival.std(ddof=1) / interarrival.mean(),
    },
    {
        "process": "poisson",
        "mean_count_60s": poisson_counts_60.mean(),
        "var_count_60s": poisson_counts_60.var(ddof=1),
        "dispersion_index": poisson_counts_60.var(ddof=1) / poisson_counts_60.mean(),
        "lag1_count_autocorr": poisson_counts_60.autocorr(lag=1),
        "interarrival_cv": poisson_events["time"].diff().dropna().std(ddof=1) / poisson_events["time"].diff().dropna().mean(),
    },
])

clustering_report

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(hawkes_counts_60, bins=40, alpha=0.6, label="Hawkes")
plt.hist(poisson_counts_60, bins=40, alpha=0.6, label="Poisson")
plt.title("60-Second Event Count Distribution")
plt.xlabel("Count")
plt.ylabel("Frequency")
plt.legend()
plt.show()

## 10. Simulate toy L1 mid-price and order-flow impact

We convert order flow into a toy mid-price path.

Buy events push price up.

Sell events push price down.

Impact has:

- temporary component;
- permanent component;
- noise;
- bid/ask spread.

This is not a production price model. It is a microstructure diagnostic.

In [ ]:
def simulate_mid_price_from_order_flow(events, config):
    rng = np.random.default_rng(config.seed + 888)
    rows = []
    mid = config.initial_mid
    temporary_state = 0.0
    last_t = 0.0
    decay_k = np.log(2) / 30.0

    for _, event in events.iterrows():
        t = event["time"]
        dt = t - last_t
        temporary_state *= np.exp(-decay_k * dt)

        side = 1 if event["event_name"] == "BUY" else -1

        permanent_move = side * config.permanent_impact_ticks * config.price_tick
        temporary_state += side * config.temporary_impact_ticks * config.price_tick

        noise = rng.normal(0, config.price_tick * 0.10)
        mid = mid + permanent_move + temporary_state + noise

        bid = mid - config.spread_ticks * config.price_tick / 2
        ask = mid + config.spread_ticks * config.price_tick / 2

        rows.append({
            "event_id": event["event_id"],
            "time": t,
            "event_name": event["event_name"],
            "side": side,
            "mid": mid,
            "bid": bid,
            "ask": ask,
            "temporary_state": temporary_state,
            "signed_event": side,
        })

        last_t = t

    return pd.DataFrame(rows)

l1_toy = simulate_mid_price_from_order_flow(events, config)

l1_toy.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(l1_toy["time"], l1_toy["mid"])
plt.title("Toy Mid-Price from Hawkes Order Flow")
plt.xlabel("Time, seconds")
plt.ylabel("Mid price")
plt.show()

## 11. Order-flow imbalance

A simple rolling event imbalance:

$$
OFI_t = \frac{N_B(t-W,t)-N_S(t-W,t)} {N_B(t-W,t)+N_S(t-W,t)}
$$

This is a crude but useful microstructure feature.

In [ ]:
def rolling_order_flow_imbalance(events, window_seconds):
    event_times = events["time"].to_numpy()
    sides = np.where(events["event_name"].to_numpy() == "BUY", 1, -1)

    rows = []
    left = 0
    for right, t in enumerate(event_times):
        while event_times[left] < t - window_seconds:
            left += 1

        window_sides = sides[left:right + 1]
        buys = np.sum(window_sides > 0)
        sells = np.sum(window_sides < 0)
        total = buys + sells
        imbalance = (buys - sells) / total if total > 0 else 0.0

        rows.append({
            "event_id": events["event_id"].iloc[right],
            "time": t,
            "buys_window": buys,
            "sells_window": sells,
            "total_window": total,
            "order_flow_imbalance": imbalance,
        })

    return pd.DataFrame(rows)

ofi = rolling_order_flow_imbalance(events, config.imbalance_window_seconds)
ofi_price = ofi.merge(l1_toy[["event_id", "mid"]], on="event_id", how="left")
ofi_price["future_mid_change_50_events"] = ofi_price["mid"].shift(-50) - ofi_price["mid"]

ofi_price.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(ofi_price["time"], ofi_price["order_flow_imbalance"])
plt.axhline(0, linestyle="--")
plt.title("Rolling Order-Flow Imbalance")
plt.xlabel("Time, seconds")
plt.ylabel("OFI")
plt.show()

ofi_corr = ofi_price[["order_flow_imbalance", "future_mid_change_50_events"]].dropna().corr().iloc[0, 1]
pd.Series({"ofi_future_price_corr_50_events": ofi_corr})

## 12. Intensity imbalance as short-horizon signal

Hawkes intensity imbalance can be interpreted as a forecast of future order-flow pressure:

$$
\frac{\lambda_B-\lambda_S}{\lambda_B+\lambda_S}
$$

We compare it to future signed order flow.

In [ ]:
def future_signed_flow(events, horizon_events=50):
    sides = np.where(events["event_name"].to_numpy() == "BUY", 1, -1)
    future = []
    for i in range(len(sides)):
        end = min(len(sides), i + 1 + horizon_events)
        if i + 1 >= end:
            future.append(np.nan)
        else:
            future.append(np.sum(sides[i + 1:end]))
    out = events[["event_id", "time"]].copy()
    out["future_signed_flow"] = future
    return out

future_flow = future_signed_flow(events, horizon_events=50)

event_intensities = events[["event_id", "time", "lambda_buy_before", "lambda_sell_before"]].copy()
event_intensities["intensity_imbalance"] = (
    event_intensities["lambda_buy_before"] - event_intensities["lambda_sell_before"]
) / (event_intensities["lambda_buy_before"] + event_intensities["lambda_sell_before"])

intensity_signal = event_intensities.merge(future_flow, on=["event_id", "time"], how="left").dropna()
intensity_signal_corr = intensity_signal[["intensity_imbalance", "future_signed_flow"]].corr().iloc[0, 1]

pd.Series({"intensity_imbalance_future_signed_flow_corr": intensity_signal_corr})

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(intensity_signal["intensity_imbalance"], intensity_signal["future_signed_flow"], alpha=0.15)
plt.axhline(0, linestyle="--")
plt.axvline(0, linestyle="--")
plt.title("Intensity Imbalance vs Future Signed Flow")
plt.xlabel("Intensity imbalance")
plt.ylabel("Future signed flow, next 50 events")
plt.show()

## 13. Univariate Hawkes likelihood

For a univariate exponential Hawkes process:

$$
\lambda(t)=\mu+\sum_{t_i<t}\alpha e^{-\beta(t-t_i)}
$$

Log-likelihood on $[0,T]$:

$$
\begin{aligned}
\mathcal{L} &= \sum_i \log \lambda(t_i) \\
&\quad - \int_0^T \lambda(t)dt
\end{aligned}
$$

For exponential kernels:

$$
\begin{aligned}
\int_0^T \lambda(t)dt &= \mu T \\
&\quad + \sum_i \frac{\alpha}{\beta} \Big( 1-e^{-\beta(T-t_i)} \Big)
\end{aligned}
$$

We fit separate univariate Hawkes models to buy and sell arrivals.

In [ ]:
def univariate_hawkes_loglik(event_times, T, mu, alpha, beta):
    event_times = np.asarray(event_times, dtype=float)

    if mu <= 0 or alpha < 0 or beta <= 0 or alpha / beta >= 0.999:
        return -np.inf

    intensities = np.zeros(len(event_times))
    r = 0.0
    last_t = 0.0

    for i, t in enumerate(event_times):
        dt = t - last_t
        r *= np.exp(-beta * dt)
        intensities[i] = mu + alpha * r
        r += 1.0
        last_t = t

    if np.any(intensities <= 0):
        return -np.inf

    integral = mu * T + (alpha / beta) * np.sum(1.0 - np.exp(-beta * (T - event_times)))
    return float(np.sum(np.log(intensities)) - integral)

def fit_univariate_hawkes(event_times, T):
    event_times = np.asarray(event_times, dtype=float)
    rate = len(event_times) / T

    if SCIPY_AVAILABLE:
        def objective(theta):
            log_mu, log_alpha, log_beta = theta
            mu = np.exp(log_mu)
            alpha = np.exp(log_alpha)
            beta = np.exp(log_beta)
            ll = univariate_hawkes_loglik(event_times, T, mu, alpha, beta)
            if not np.isfinite(ll):
                return 1e12
            return -ll

        x0 = np.log([max(rate * 0.6, 1e-6), max(rate * 0.5, 1e-6), 1.0])
        result = minimize(objective, x0=x0, method="Nelder-Mead", options={"maxiter": 3000})

        mu, alpha, beta = np.exp(result.x)
        ll = univariate_hawkes_loglik(event_times, T, mu, alpha, beta)

        return {
            "mu": mu,
            "alpha": alpha,
            "beta": beta,
            "branching_ratio": alpha / beta,
            "loglik": ll,
            "success": bool(result.success),
            "method": "scipy_nelder_mead",
        }

    # Fallback: coarse grid.
    best = None
    for mu in np.linspace(rate * 0.2, rate * 1.2, 8):
        for branching in np.linspace(0.05, 0.85, 9):
            for beta in np.linspace(0.3, 3.0, 10):
                alpha = branching * beta
                ll = univariate_hawkes_loglik(event_times, T, mu, alpha, beta)
                if best is None or ll > best["loglik"]:
                    best = {
                        "mu": mu,
                        "alpha": alpha,
                        "beta": beta,
                        "branching_ratio": alpha / beta,
                        "loglik": ll,
                        "success": True,
                        "method": "coarse_grid",
                    }
    return best

buy_times = events.loc[events["event_name"] == "BUY", "time"].to_numpy()
sell_times = events.loc[events["event_name"] == "SELL", "time"].to_numpy()

buy_fit = fit_univariate_hawkes(buy_times, config.horizon_seconds)
sell_fit = fit_univariate_hawkes(sell_times, config.horizon_seconds)

univariate_fit_report = pd.DataFrame([
    {"side": "BUY", **buy_fit},
    {"side": "SELL", **sell_fit},
])

univariate_fit_report

## 14. Compare Hawkes to Poisson likelihood

A univariate Poisson process has MLE:

$$
\hat \lambda = \frac{N(T)}{T}
$$

and log-likelihood:

$$
\sum_i \log \hat\lambda - \hat\lambda T
$$

We compare AIC-style scores.

In [ ]:
def poisson_loglik(event_times, T):
    event_times = np.asarray(event_times)
    rate = len(event_times) / T
    if rate <= 0:
        return -np.inf
    return float(len(event_times) * np.log(rate) - rate * T)

poisson_buy_ll = poisson_loglik(buy_times, config.horizon_seconds)
poisson_sell_ll = poisson_loglik(sell_times, config.horizon_seconds)

model_likelihood_comparison = pd.DataFrame([
    {
        "side": "BUY",
        "model": "poisson",
        "loglik": poisson_buy_ll,
        "n_params": 1,
        "AIC": 2 * 1 - 2 * poisson_buy_ll,
    },
    {
        "side": "BUY",
        "model": "hawkes",
        "loglik": buy_fit["loglik"],
        "n_params": 3,
        "AIC": 2 * 3 - 2 * buy_fit["loglik"],
    },
    {
        "side": "SELL",
        "model": "poisson",
        "loglik": poisson_sell_ll,
        "n_params": 1,
        "AIC": 2 * 1 - 2 * poisson_sell_ll,
    },
    {
        "side": "SELL",
        "model": "hawkes",
        "loglik": sell_fit["loglik"],
        "n_params": 3,
        "AIC": 2 * 3 - 2 * sell_fit["loglik"],
    },
])

model_likelihood_comparison.sort_values(["side", "AIC"])

## 15. Time-rescaling residual diagnostics

For a correctly specified point process, transformed event times:

$$
\tau_i = \int_{t_{i-1}}^{t_i}\lambda(s)ds
$$

should be approximately iid exponential with mean 1.

Then:

$$
u_i = 1-e^{-\tau_i}
$$

should be approximately uniform on $[0,1]$.

In [ ]:
def hawkes_residuals_univariate(event_times, T, mu, alpha, beta):
    event_times = np.asarray(event_times, dtype=float)
    residuals = []
    last_t = 0.0
    history = []

    for t in event_times:
        # Integral from last_t to t.
        dt_integral = mu * (t - last_t)
        excitation_integral = 0.0

        for ti in history:
            excitation_integral += (alpha / beta) * (
                np.exp(-beta * (last_t - ti)) - np.exp(-beta * (t - ti))
            )

        tau = dt_integral + excitation_integral
        residuals.append(tau)

        history.append(t)
        last_t = t

    residuals = np.array(residuals)
    uniforms = 1.0 - np.exp(-residuals)

    return pd.DataFrame({
        "residual_exponential": residuals,
        "residual_uniform": uniforms,
    })

buy_residuals = hawkes_residuals_univariate(
    buy_times,
    config.horizon_seconds,
    buy_fit["mu"],
    buy_fit["alpha"],
    buy_fit["beta"],
)

sell_residuals = hawkes_residuals_univariate(
    sell_times,
    config.horizon_seconds,
    sell_fit["mu"],
    sell_fit["alpha"],
    sell_fit["beta"],
)

residual_summary = pd.DataFrame([
    {
        "side": "BUY",
        "mean_exp_residual": buy_residuals["residual_exponential"].mean(),
        "var_exp_residual": buy_residuals["residual_exponential"].var(),
        "mean_uniform_residual": buy_residuals["residual_uniform"].mean(),
    },
    {
        "side": "SELL",
        "mean_exp_residual": sell_residuals["residual_exponential"].mean(),
        "var_exp_residual": sell_residuals["residual_exponential"].var(),
        "mean_uniform_residual": sell_residuals["residual_uniform"].mean(),
    },
])

residual_summary

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(buy_residuals["residual_uniform"], bins=config.residual_bins, alpha=0.6, label="BUY")
plt.hist(sell_residuals["residual_uniform"], bins=config.residual_bins, alpha=0.6, label="SELL")
plt.title("Time-Rescaled Uniform Residuals")
plt.xlabel("u")
plt.ylabel("Count")
plt.legend()
plt.show()

## 16. Moment-style multivariate branching diagnostics

Full multivariate Hawkes MLE is more involved.

Here we use the known simulation parameters and fitted univariate estimates to build practical diagnostics:

- true branching matrix;
- implied same-side branching from univariate fits;
- spectral radius;
- near-critical risk.

In production, this should be replaced with full multivariate maximum likelihood or specialised point-process libraries.

In [ ]:
estimated_branching_proxy = pd.DataFrame(
    [
        [buy_fit["branching_ratio"], np.nan],
        [np.nan, sell_fit["branching_ratio"]],
    ],
    index=["BUY", "SELL"],
    columns=["BUY", "SELL"],
)

true_branching = pd.DataFrame(
    params["branching"],
    index=["BUY", "SELL"],
    columns=["BUY", "SELL"],
)

branching_diagnostics = pd.DataFrame([{
    "true_spectral_radius": params["spectral_radius"],
    "buy_univariate_branching": buy_fit["branching_ratio"],
    "sell_univariate_branching": sell_fit["branching_ratio"],
    "max_univariate_branching": max(buy_fit["branching_ratio"], sell_fit["branching_ratio"]),
    "true_stable": params["spectral_radius"] < 1,
    "univariate_proxy_stable": max(buy_fit["branching_ratio"], sell_fit["branching_ratio"]) < 1,
}])

true_branching, estimated_branching_proxy, branching_diagnostics

## 17. Intensity forecasting

Given current intensity, the expected future intensity decays toward baseline unless new events arrive.

For univariate Hawkes:

$$
E[\lambda(t+h)\mid \mathcal{F}_t] \approx \mu + (\lambda(t)-\mu)e^{-\beta h}
$$

This is useful for execution:

- if intensity is high, near-term order-flow risk is elevated;
- if buy intensity dominates, buying pressure may persist;
- if sell intensity dominates, selling pressure may persist.

In [ ]:
def forecast_intensity_from_latest(events, params, horizons):
    latest_time = events["time"].iloc[-1]
    latest = compute_intensity_grid(events, params, latest_time, grid_step=max(latest_time, 1.0)).iloc[-1]

    mu = params["mu"]
    # Use average beta by target side for simple forecast.
    beta_buy = params["beta"][0].mean()
    beta_sell = params["beta"][1].mean()

    rows = []
    for h in horizons:
        lambda_buy = mu[0] + (latest["lambda_buy"] - mu[0]) * np.exp(-beta_buy * h)
        lambda_sell = mu[1] + (latest["lambda_sell"] - mu[1]) * np.exp(-beta_sell * h)
        rows.append({
            "horizon_seconds": h,
            "forecast_lambda_buy": lambda_buy,
            "forecast_lambda_sell": lambda_sell,
            "forecast_total_intensity": lambda_buy + lambda_sell,
            "forecast_intensity_imbalance": (lambda_buy - lambda_sell) / max(lambda_buy + lambda_sell, 1e-12),
        })

    return pd.DataFrame(rows)

intensity_forecast = forecast_intensity_from_latest(
    events,
    params,
    horizons=np.array([1, 2, 5, 10, 30, 60, 120, 300]),
)

intensity_forecast

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(intensity_forecast["horizon_seconds"], intensity_forecast["forecast_lambda_buy"], marker="o", label="BUY")
plt.plot(intensity_forecast["horizon_seconds"], intensity_forecast["forecast_lambda_sell"], marker="o", label="SELL")
plt.title("Intensity Forecast from Latest State")
plt.xlabel("Horizon, seconds")
plt.ylabel("Forecast intensity")
plt.legend()
plt.show()

## 18. Execution-risk use case

A trader executing a buy order may want to avoid moments when:

- buy intensity is already high;
- sell intensity is high and adverse selection is likely;
- total intensity is elevated;
- order-flow imbalance is unstable.

We create a simple execution-risk score:

$$
\begin{aligned}
RiskScore &= z(\lambda_B+\lambda_S) \\
&\quad + |z(imbalance)| \\
&\quad + z(count\_60s)
\end{aligned}
$$

In [ ]:
def execution_risk_score(intensity_grid, hawkes_counts_60):
    # Map 60s counts to intensity grid time.
    count_df = hawkes_counts_60.rename("count_60s").reset_index().rename(columns={"index": "bucket_start"})
    grid = intensity_grid.copy()
    grid["bucket_start"] = (np.floor(grid["time"] / 60.0) * 60.0).astype(float)
    grid = grid.merge(count_df, on="bucket_start", how="left")
    grid["count_60s"] = grid["count_60s"].fillna(0)

    def zscore(x):
        s = x.std(ddof=1)
        return (x - x.mean()) / s if s > 0 else x * 0

    grid["z_total_intensity"] = zscore(grid["total_intensity"])
    grid["z_abs_imbalance"] = zscore(grid["intensity_imbalance"].abs())
    grid["z_count_60s"] = zscore(grid["count_60s"])

    grid["execution_risk_score"] = (
        grid["z_total_intensity"]
        + grid["z_abs_imbalance"]
        + grid["z_count_60s"]
    )

    grid["risk_bucket"] = pd.qcut(grid["execution_risk_score"], q=4, labels=["low", "medium", "high", "extreme"])

    return grid

execution_risk = execution_risk_score(intensity_grid, hawkes_counts_60)

execution_risk.tail()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(execution_risk["time"], execution_risk["execution_risk_score"])
plt.axhline(execution_risk["execution_risk_score"].quantile(0.75), linestyle="--", label="75th percentile")
plt.title("Execution Risk Score from Hawkes Order Flow")
plt.xlabel("Time, seconds")
plt.ylabel("Risk score")
plt.legend()
plt.show()

## 19. Stress scenario: near-critical Hawkes process

When branching ratio approaches 1, order flow becomes extremely clustered.

We compare the base process to a near-critical parameter set.

In [ ]:
def make_near_critical_config(config):
    return HawkesOrderFlowConfig(
        horizon_seconds=config.horizon_seconds,
        seed=config.seed + 2024,
        baseline_buy=config.baseline_buy,
        baseline_sell=config.baseline_sell,
        alpha_bb=config.alpha_bb * 2.5,
        alpha_bs=config.alpha_bs * 2.0,
        alpha_sb=config.alpha_sb * 2.0,
        alpha_ss=config.alpha_ss * 2.5,
        beta_bb=config.beta_bb,
        beta_bs=config.beta_bs,
        beta_sb=config.beta_sb,
        beta_ss=config.beta_ss,
        max_events=config.max_events,
        intensity_cap=config.intensity_cap,
        output_subdir=config.output_subdir,
    )

stress_config = make_near_critical_config(config)
stress_params = parameter_matrices(stress_config)
stress_events = simulate_bivariate_hawkes(stress_config, stress_params)
stress_flow_summary, stress_interarrival = event_flow_diagnostics(stress_events, stress_config)

stress_counts_60 = count_by_window(stress_events, 60.0)

stress_comparison = pd.DataFrame([
    {
        "scenario": "base",
        "spectral_radius": params["spectral_radius"],
        "total_events": len(events),
        "events_per_second": len(events) / config.horizon_seconds,
        "dispersion_60s": hawkes_counts_60.var(ddof=1) / hawkes_counts_60.mean(),
        "lag1_count_autocorr": hawkes_counts_60.autocorr(1),
    },
    {
        "scenario": "near_critical",
        "spectral_radius": stress_params["spectral_radius"],
        "total_events": len(stress_events),
        "events_per_second": len(stress_events) / stress_config.horizon_seconds,
        "dispersion_60s": stress_counts_60.var(ddof=1) / stress_counts_60.mean(),
        "lag1_count_autocorr": stress_counts_60.autocorr(1),
    },
])

stress_comparison

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(hawkes_counts_60, bins=40, alpha=0.6, label="base")
plt.hist(stress_counts_60, bins=40, alpha=0.6, label="near critical")
plt.title("60s Count Distribution: Base vs Near-Critical")
plt.xlabel("Events per 60s")
plt.ylabel("Frequency")
plt.legend()
plt.show()

## 20. Governance flags

A Hawkes order-flow model should be reviewed if:

1. branching ratio is near 1;
2. fitted univariate branching is near 1;
3. residual diagnostics are poor;
4. event clustering is extreme;
5. intensity forecasts are unstable;
6. execution risk score shows frequent extremes;
7. model is calibrated on synthetic rather than real timestamped order data.

In [ ]:
uniform_mean_deviation = max(
    abs(buy_residuals["residual_uniform"].mean() - 0.5),
    abs(sell_residuals["residual_uniform"].mean() - 0.5),
)

dispersion_index = clustering_report.loc[clustering_report["process"] == "hawkes", "dispersion_index"].iloc[0]
poisson_dispersion = clustering_report.loc[clustering_report["process"] == "poisson", "dispersion_index"].iloc[0]
extreme_risk_fraction = (execution_risk["risk_bucket"] == "extreme").mean()

governance_flags = pd.DataFrame([{
    "true_spectral_radius": params["spectral_radius"],
    "max_univariate_branching": branching_diagnostics["max_univariate_branching"].iloc[0],
    "uniform_residual_mean_deviation": uniform_mean_deviation,
    "hawkes_dispersion_index_60s": dispersion_index,
    "poisson_dispersion_index_60s": poisson_dispersion,
    "dispersion_ratio_vs_poisson": dispersion_index / poisson_dispersion if poisson_dispersion > 0 else np.nan,
    "extreme_execution_risk_fraction": extreme_risk_fraction,
    "near_critical_spectral_radius": stress_params["spectral_radius"],
    "flag_true_spectral_radius_above_0_8": bool(params["spectral_radius"] > 0.80),
    "flag_univariate_branching_above_0_8": bool(branching_diagnostics["max_univariate_branching"].iloc[0] > 0.80),
    "flag_residual_mean_far_from_uniform": bool(uniform_mean_deviation > 0.10),
    "flag_extreme_clustering": bool(dispersion_index / poisson_dispersion > 2.0) if poisson_dispersion > 0 else False,
    "flag_near_critical_stress_unstable": bool(stress_params["spectral_radius"] > 0.95),
    "flag_synthetic_data_not_real_order_feed": True,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_true_spectral_radius_above_0_8",
        "flag_univariate_branching_above_0_8",
        "flag_residual_mean_far_from_uniform",
        "flag_extreme_clustering",
        "flag_near_critical_stress_unstable",
        "flag_synthetic_data_not_real_order_feed",
    ]
].any(axis=1)

governance_flags

## 21. Audit checks

Numerical and process checks:

1. simulated events are time-ordered;
2. intensities are positive;
3. branching matrix is finite;
4. stability diagnostic is finite;
5. fitted parameters are finite;
6. residuals are finite;
7. execution risk score is finite;
8. governance flags exist.

In [ ]:
audit_rows = []

time_ordered = bool(events["time"].is_monotonic_increasing)
audit_rows.append({
    "check": "events_time_ordered",
    "value": time_ordered,
    "passed": time_ordered,
})

intensities_positive = bool(
    (events[["lambda_buy_before", "lambda_sell_before", "total_intensity_before"]] > 0).all().all()
)
audit_rows.append({
    "check": "event_intensities_positive",
    "value": intensities_positive,
    "passed": intensities_positive,
})

branching_finite = bool(np.isfinite(params["branching"]).all())
audit_rows.append({
    "check": "branching_matrix_finite",
    "value": branching_finite,
    "passed": branching_finite,
})

spectral_radius_finite = bool(np.isfinite(params["spectral_radius"]))
audit_rows.append({
    "check": "spectral_radius_finite",
    "value": spectral_radius_finite,
    "passed": spectral_radius_finite,
})

fit_finite = bool(np.isfinite(univariate_fit_report[["mu", "alpha", "beta", "branching_ratio", "loglik"]].to_numpy()).all())
audit_rows.append({
    "check": "univariate_fit_outputs_finite",
    "value": fit_finite,
    "passed": fit_finite,
})

residuals_finite = bool(
    np.isfinite(buy_residuals.to_numpy()).all()
    and np.isfinite(sell_residuals.to_numpy()).all()
)
audit_rows.append({
    "check": "residuals_finite",
    "value": residuals_finite,
    "passed": residuals_finite,
})

risk_score_finite = bool(np.isfinite(execution_risk[["execution_risk_score"]].to_numpy()).all())
audit_rows.append({
    "check": "execution_risk_score_finite",
    "value": risk_score_finite,
    "passed": risk_score_finite,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 22. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

events.to_csv(output_dir / "hawkes_events.csv", index=False)
poisson_events.to_csv(output_dir / "poisson_events.csv", index=False)
intensity_grid.to_csv(output_dir / "intensity_grid.csv", index=False)
flow_summary.to_csv(output_dir / "flow_summary.csv", index=False)
clustering_report.to_csv(output_dir / "clustering_report.csv", index=False)
l1_toy.to_csv(output_dir / "toy_l1_midprice.csv", index=False)
ofi_price.to_csv(output_dir / "order_flow_imbalance.csv", index=False)
intensity_signal.to_csv(output_dir / "intensity_signal.csv", index=False)
univariate_fit_report.to_csv(output_dir / "univariate_hawkes_fit.csv", index=False)
model_likelihood_comparison.to_csv(output_dir / "model_likelihood_comparison.csv", index=False)
buy_residuals.to_csv(output_dir / "buy_residuals.csv", index=False)
sell_residuals.to_csv(output_dir / "sell_residuals.csv", index=False)
residual_summary.to_csv(output_dir / "residual_summary.csv", index=False)
branching_report.to_csv(output_dir / "true_branching_matrix.csv")
branching_diagnostics.to_csv(output_dir / "branching_diagnostics.csv", index=False)
intensity_forecast.to_csv(output_dir / "intensity_forecast.csv", index=False)
execution_risk.to_csv(output_dir / "execution_risk_score.csv", index=False)
stress_events.to_csv(output_dir / "near_critical_hawkes_events.csv", index=False)
stress_comparison.to_csv(output_dir / "stress_comparison.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "hawkes_process_order_flow_outputs",
    "schema_version": "hawkes_process_order_flow_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "scipy_available": SCIPY_AVAILABLE,
    "true_branching_matrix": params["branching"].tolist(),
    "true_spectral_radius": params["spectral_radius"],
    "flow_summary": flow_summary.to_dict(orient="records"),
    "univariate_fit_report": univariate_fit_report.to_dict(orient="records"),
    "model_likelihood_comparison": model_likelihood_comparison.to_dict(orient="records"),
    "residual_summary": residual_summary.to_dict(orient="records"),
    "branching_diagnostics": branching_diagnostics.to_dict(orient="records"),
    "stress_comparison": stress_comparison.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook simulates bivariate buy/sell Hawkes order flow.",
        "The Hawkes process is compared against a Poisson baseline.",
        "Toy L1 mid-price dynamics are generated from signed order flow.",
        "Univariate exponential Hawkes models are fitted separately to buy and sell arrivals.",
        "Time-rescaling residual diagnostics are included.",
        "Intensity imbalance and order-flow imbalance are tested as short-horizon features.",
        "A near-critical stress scenario illustrates clustered order-flow risk.",
        "The data is synthetic and should be replaced with timestamped market/order data for production research."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "hawkes_events.csv", output_dir / "univariate_hawkes_fit.csv", output_dir / "execution_risk_score.csv", output_dir / "governance_flags.csv", manifest_path

## 23. Practical checklist for Hawkes order-flow modelling

Before using a Hawkes model in execution research:

1. **Use timestamped event data.**
2. **Define event types precisely.**
3. **Separate market orders, limit orders, cancellations, and quote updates.**
4. **Check clustering against a Poisson baseline.**
5. **Estimate branching ratios.**
6. **Check spectral radius stability.**
7. **Use residual time-change diagnostics.**
8. **Validate intensity forecasts out of sample.**
9. **Stress near-critical regimes.**
10. **Do not confuse excitation with causality.**

## 24. Limitations

### Synthetic event data

This notebook uses synthetic event times. Real calibration requires exchange timestamps or high-quality vendor data.

### Simplified multivariate simulation

The simulator is designed for clarity, not high-performance production.

### Univariate calibration only

Full multivariate Hawkes MLE is more complex. This notebook fits buy and sell univariate Hawkes models separately as diagnostics.

### Toy price impact

The mid-price model is illustrative and not a production price formation model.

### No queue or order book depth

The notebook does not model limit order book queues, depth, cancellations, or hidden liquidity.

### No causality claim

Self-excitation does not prove causal triggering. It is a statistical dependence model.

## 25. Research frontier and extensions

Important extensions include:

1. full multivariate Hawkes maximum likelihood;
2. marked Hawkes processes with order size;
3. nonparametric kernel estimation;
4. regime-switching Hawkes processes;
5. mutually exciting trades and quote updates;
6. Hawkes-driven queue models;
7. Hawkes process with price impact;
8. neural point processes;
9. L1 bid/ask Hawkes calibration;
10. futures order-flow clustering with day and night sessions.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_05_order_management_system_simulator**  
   Use clustered order flow to test order handling.

2. **06_06_execution_algorithms_twap_vwap_pov**  
   Adapt execution rates to order-flow intensity.

3. **06_07_latency_and_queue_position_model**  
   Model fill risk under clustered events.

4. **06_08_microprice_and_short_horizon_alpha**  
   Convert order-flow and intensity imbalance into microprice signals.

5. **06_11_l1_bid_ask_backtest_execution_model**  
   Use L1 bid/ask data for realistic event-driven fills.

## 27. Summary

This notebook implemented Hawkes process order-flow modelling.

It showed how to:

1. define univariate and bivariate Hawkes processes;
2. compute branching ratios and spectral radius;
3. simulate clustered buy/sell order arrivals;
4. compare Hawkes against a Poisson baseline;
5. compute order-flow clustering diagnostics;
6. simulate a toy L1 mid-price from signed flow;
7. compute rolling order-flow imbalance;
8. use intensity imbalance as a short-horizon signal;
9. fit univariate exponential Hawkes models;
10. compare Hawkes and Poisson likelihoods;
11. run time-rescaling residual diagnostics;
12. forecast intensities;
13. build an execution-risk score;
14. stress-test near-critical order flow;
15. create governance flags and audit checks;
16. save outputs and manifests.

The key computational takeaway:

> Hawkes models represent event clustering by making intensity jump after events and decay over time.

The key financial takeaway:

> Clustered order flow can turn execution risk from a smooth average-cost problem into a burst-risk problem.

## 28. Further reading

- Hawkes, "Spectra of Some Self-Exciting and Mutually Exciting Point Processes."
- Bacry, Mastromatteo and Muzy on Hawkes processes in finance.
- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- Abergel et al., *Limit Order Books.*
- Daley and Vere-Jones, *An Introduction to the Theory of Point Processes.*
- Laub, Taimre and Pollett, "Hawkes Processes."